# Decision Planner Comparison

Colab notebook for comparing seven decision planning strategies on a small CICIDS test sample.

In [ ]:
!pip install numpy scikit-learn transformers -q

In [ ]:
!git clone https://github.com/Lawapaul/AI_Agentic_DL.git

import sys
sys.path.append('/content/AI_Agentic_DL')

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import numpy as np

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed/'

X = np.load(path + 'X_test.npy')[:20]
y = np.load(path + 'y_test.npy')[:20]

X = X.reshape(X.shape[0], -1)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)

preds = model.predict(X)
probs = model.predict_proba(X)

In [ ]:
from decision_planner.rule_based import RuleBasedPlanner
from decision_planner.confidence_based import ConfidenceBasedPlanner
from decision_planner.risk_based import RiskBasedPlanner
from decision_planner.hybrid import HybridPlanner
from decision_planner.rl_planner import RLPlanner
from decision_planner.llm_planner import LLMPlanner
from decision_planner.policy_based import PolicyBasedPlanner

In [ ]:
rule = RuleBasedPlanner()
conf = ConfidenceBasedPlanner()
risk = RiskBasedPlanner()
hybrid = HybridPlanner()
rl = RLPlanner()
llm = LLMPlanner()
policy = PolicyBasedPlanner()

In [ ]:
results = []

for i in range(10):
    pred = int(preds[i])
    conf_score = float(max(probs[i]))

    attack = 'Attack' if pred != 0 else 'Normal Traffic'
    severity = 'High' if conf_score > 0.8 else 'Medium' if conf_score > 0.6 else 'Low'

    results.append({
        'index': i,
        'attack': attack,
        'confidence': conf_score,
        'severity': severity,
        'rule': rule.decide(attack, conf_score, severity),
        'confidence_based': conf.decide(attack, conf_score, severity),
        'risk': risk.decide(attack, conf_score, severity),
        'hybrid': hybrid.decide(attack, conf_score, severity),
        'rl': rl.decide(attack, conf_score, severity),
        'llm': llm.decide(attack, conf_score, severity),
        'policy': policy.decide(attack, conf_score, severity)
    })

    print(results[-1])

In [ ]:
import json
import os

save_path = '/content/AI_Agentic_DL/experiments/results'
os.makedirs(save_path, exist_ok=True)

with open(os.path.join(save_path, 'decision_results_all.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('Saved successfully')